# MERGE between categories based on filters
Unisce i merge basati su categoria (volumi, biomarcatori, cofattori, scale)

## Import

In [3]:
from data_model.DataCleaner import *
from dl_client import DatalakeClient
from data_model.manage_excel_support_file import *
from data_model.MergerTools import *
import pandas as pd
import os

client = DatalakeClient()
mergeTools = MergerTools()

## Download files fom DL

In [4]:
file_codes =[]
search = client.query_files(
    query={'custom.level' : 'merged', 'custom.source' : 'ADNI', 'custom.file_code': file_codes})

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True)

print(len(zip_files))

Exception: Query failed: No files match the query criteria

In [ ]:
dfs = {}
df_names = {}
df_code = []
for idx, (file_name, df_raw) in enumerate(zip_files.items()):
    df_copy = df_raw.copy(deep=True)
    # get the sub_df focusing on the category chosen
    # get the common columns among all the dfs
    if idx == 0:
        dfs_columns = set(df_copy.columns)
    else: 
        dfs_columns &= set(df_copy.columns)
    # Ensure EXAMDATE in date format and correct order of the dfs
    df_copy['EXAMDATE'] = pd.to_datetime(df_copy['EXAMDATE'])
    df_copy = df_copy.sort_values(by=['RID', 'EXAMDATE']).reset_index(drop=True)
    if 'FSVERSION' in df_copy.columns:
        df_copy['FSVERSION'] = df_copy['FSVERSION'].astype(str)
    # aggiornamento liste e dizionari
    dfs[f"df_{idx}"] = df_copy  
    df_names[f"df_{idx}"] = file_name 
    df_code.append(f"df_{idx}")
    # definizione variabile df
    globals()[f"df_{idx}"] = df_copy
    print(idx, '--->', file_name)

time_buffer = pd.Timedelta(days=80)

### Confronto stessi RID  ==> RID - EXAMDATE identici tra file

In [ ]:
subj_matrix = mergeTools.matrix_match(dfs, df_names, df_code, columns_list=['RID'])
print("righe con stessi ####### RID:")
display(subj_matrix)
        
subj_date_matrix = mergeTools.matrix_match(dfs, df_names, df_code, columns_list=['RID', 'EXAMDATE'], time_buffer=time_buffer)
print("righe con stessi ####### RID-EXAMDATE: --> time_buffer=", time_buffer)
display(subj_date_matrix)

# Merge pipeline

In [ ]:
base = 'df_4'
df_base = dfs[base].copy(deep=True)                           #sembra un errore ma questi df sono definiti
idx_add = [ 'df_6', 'df_7', 'df_8', 'df_9','df_1', 'df_2', 'df_3', 'df_5', 'df_0']
merge_contains = [base]
sub_with_match = set()
time_buffer = pd.Timedelta(days=80)
i = 0

## ADD df by df
Con attenzione merge e dove RID - EXAMDATE sono uguali e filtrati per:
- Volumi: FSVERSION
- Biomarcatori: METHOD